# Week 2 — Train ECAPA-TDNN From Scratch (100 speakers) + Compare to Baseline

**Scope decisions (locked in):**
- ~100 speakers sampled from the 1,211-speaker VoxCeleb1 dev set (random init, no pretrained weights — genuinely from scratch)
- Smaller ECAPA-TDNN (channels=512 instead of the full recipe's 1024) to keep it tractable on a free-tier T4
- AAM-Softmax loss (angular margin), matching the standard SpeechBrain recipe approach, just at reduced scale
- Evaluated with the **same protocol as the pretrained baseline** (same 40 held-out test speakers for verification, same `iden_split.txt` identification task) for a fair scratch-vs-pretrained comparison

At this scale, expect epochs to take a few minutes each on a T4 (rough estimate — watch the first epoch's timing and adjust `N_EPOCHS` if needed). This should comfortably fit in a single 12-hour Kaggle session, unlike full-scale training.

Make sure GPU + Internet are ON (Internet only needed for the `pip install`, not for training itself — no pretrained weights are downloaded this time).

In [ ]:
!pip install -q speechbrain


In [ ]:
import os, random, pickle, time
from pathlib import Path
from collections import defaultdict
import torch
import torch.nn as nn
import torchaudio
import torch.nn.functional as F
import numpy as np
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)
torch.manual_seed(42)
random.seed(42)


In [ ]:
# ---- Paths (same as baseline notebook) ----
path_split = Path("/kaggle/input/datasets/sabahesaraki/voxceleb-1-dataset")
path_data = Path("/kaggle/input/datasets/kryakrya")
path_data_train = path_data.joinpath("voxceleb1train/wav")   # 1,211 dev-speaker folders
path_data_test  = path_data.joinpath("voxceleb1test/wav")    # 40 test-speaker folders

VERI_TEST_PATH = path_split / "veri_test2.txt"
IDEN_SPLIT_PATH = path_split / "iden_split.txt"

def resolve_wav(rel_path: str) -> Path:
    p_test = path_data_test / rel_path
    if p_test.exists():
        return p_test
    p_train = path_data_train / rel_path
    if p_train.exists():
        return p_train
    raise FileNotFoundError(f"Could not find {rel_path}")

print("Paths OK.")


## 1. Select the 100-speaker training subset and build train/val file lists

In [ ]:
N_SPEAKERS = 100
N_VAL_PER_SPEAKER = 2  # held-out utterances per speaker, just to monitor training (not final eval)

all_train_speakers = sorted([p.name for p in path_data_train.iterdir() if p.is_dir()])
print("Total available dev speakers:", len(all_train_speakers))

random.seed(42)
selected_speakers = sorted(random.sample(all_train_speakers, N_SPEAKERS))
speaker_to_label = {spk: i for i, spk in enumerate(selected_speakers)}
print(f"Selected {len(selected_speakers)} speakers for training.")

train_files = []  # (rel_path, label)
val_files = []
for spk in tqdm(selected_speakers, desc="Listing files"):
    spk_dir = path_data_train / spk
    wav_files = list(spk_dir.rglob("*.wav"))
    rel_paths = [str(w.relative_to(path_data_train)) for w in wav_files]
    random.shuffle(rel_paths)
    val_part = rel_paths[:N_VAL_PER_SPEAKER]
    train_part = rel_paths[N_VAL_PER_SPEAKER:]
    label = speaker_to_label[spk]
    train_files.extend([(p, label) for p in train_part])
    val_files.extend([(p, label) for p in val_part])

print("Train utterances:", len(train_files))
print("Val utterances:  ", len(val_files))


## 2. Dataset — random 3-second crops, on the fly

Fixed-length crops mean no padding logic is needed and batches can be simple stacked tensors.

In [ ]:
SAMPLE_RATE = 16000
CROP_SECONDS = 3.0
CROP_SAMPLES = int(SAMPLE_RATE * CROP_SECONDS)

class VoxCelebCropDataset(Dataset):
    def __init__(self, file_label_list, root):
        self.items = file_label_list
        self.root = root

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        rel_path, label = self.items[idx]
        full_path = self.root / rel_path
        signal, fs = torchaudio.load(str(full_path))
        if fs != SAMPLE_RATE:
            signal = torchaudio.functional.resample(signal, fs, SAMPLE_RATE)
        signal = signal.mean(dim=0)  # mono

        if signal.shape[0] >= CROP_SAMPLES:
            start = random.randint(0, signal.shape[0] - CROP_SAMPLES)
            signal = signal[start:start + CROP_SAMPLES]
        else:
            # pad short utterances by repeating
            reps = CROP_SAMPLES // signal.shape[0] + 1
            signal = signal.repeat(reps)[:CROP_SAMPLES]

        return signal, label

train_ds = VoxCelebCropDataset(train_files, path_data_train)
val_ds = VoxCelebCropDataset(val_files, path_data_train)

BATCH_SIZE = 32
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, drop_last=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

# Sanity check one batch
xb, yb = next(iter(train_loader))
print("Batch shapes:", xb.shape, yb.shape)


## 3. Model — small ECAPA-TDNN (random init) + AAM-Softmax classification head

In [ ]:
from speechbrain.lobes.models.ECAPA_TDNN import ECAPA_TDNN, Classifier
from speechbrain.lobes.features import Fbank
from speechbrain.processing.features import InputNormalization
from speechbrain.nnet.losses import LogSoftmaxWrapper, AdditiveAngularMargin

N_MELS = 80
EMB_DIM = 192

compute_features = Fbank(n_mels=N_MELS).to(device)
mean_var_norm = InputNormalization(norm_type="sentence", std_norm=False).to(device)

embedding_model = ECAPA_TDNN(
    input_size=N_MELS,
    channels=[512, 512, 512, 512, 1536],   # scaled down from the full [1024,1024,1024,1024,3072]
    kernel_sizes=[5, 3, 3, 3, 1],
    dilations=[1, 2, 3, 4, 1],
    groups=[1, 1, 1, 1, 1],
    attention_channels=128,
    lin_neurons=EMB_DIM,
).to(device)

classifier = Classifier(
    input_size=EMB_DIM,
    out_neurons=N_SPEAKERS,
).to(device)

n_params = sum(p.numel() for p in embedding_model.parameters())
print(f"Embedding model params: {n_params/1e6:.2f}M")

loss_fn = LogSoftmaxWrapper(loss_fn=AdditiveAngularMargin(margin=0.2, scale=30))

def extract_features(signal_batch):
    """signal_batch: [B, T] raw waveform on device -> normalized fbank features [B, frames, n_mels]"""
    feats = compute_features(signal_batch)
    lens = torch.ones(feats.shape[0], device=device)
    feats = mean_var_norm(feats, lens)
    return feats


## 4. Training loop (with per-epoch checkpointing)

In [ ]:
N_EPOCHS = 30
LR = 0.001

params = list(embedding_model.parameters()) + list(classifier.parameters())
optimizer = torch.optim.Adam(params, lr=LR)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=LR, total_steps=N_EPOCHS * len(train_loader)
)

CKPT_DIR = Path("/kaggle/working/checkpoints")
CKPT_DIR.mkdir(exist_ok=True)

history = {"train_loss": [], "train_acc": [], "val_acc": []}

def run_epoch(loader, train=True):
    embedding_model.train(train)
    classifier.train(train)
    total_loss, total_correct, total_n = 0.0, 0, 0
    for signal, label in loader:
        signal, label = signal.to(device), label.to(device)
        with torch.set_grad_enabled(train):
            feats = extract_features(signal)
            emb = embedding_model(feats)          # [B, 1, EMB_DIM]
            logits = classifier(emb)              # [B, 1, N_SPEAKERS]
            loss = loss_fn(logits, label.unsqueeze(1))
            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
                scheduler.step()

        preds = logits.squeeze(1).argmax(dim=1)
        total_correct += (preds == label).sum().item()
        total_n += label.size(0)
        total_loss += loss.item() * label.size(0)

    return total_loss / total_n, total_correct / total_n


In [ ]:
for epoch in range(1, N_EPOCHS + 1):
    t0 = time.time()
    train_loss, train_acc = run_epoch(train_loader, train=True)
    val_loss, val_acc = run_epoch(val_loader, train=False)
    dt = time.time() - t0

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_acc"].append(val_acc)

    print(f"Epoch {epoch:02d}/{N_EPOCHS}  train_loss={train_loss:.4f}  "
          f"train_acc={train_acc*100:.1f}%  val_acc={val_acc*100:.1f}%  ({dt:.1f}s)")

    torch.save({
        'embedding_model': embedding_model.state_dict(),
        'classifier': classifier.state_dict(),
        'epoch': epoch,
        'history': history,
    }, CKPT_DIR / f"ckpt_epoch{epoch}.pt")
    # keep only the latest checkpoint to save disk space
    prev = CKPT_DIR / f"ckpt_epoch{epoch-1}.pt"
    if prev.exists():
        prev.unlink()

print("Training complete.")


In [ ]:
plt.figure(figsize=(10,4))
plt.subplot(1,2,1)
plt.plot(history["train_loss"])
plt.title("Train Loss"); plt.xlabel("Epoch"); plt.grid(alpha=0.3)

plt.subplot(1,2,2)
plt.plot(history["train_acc"], label="train")
plt.plot(history["val_acc"], label="val")
plt.title("Speaker Classification Accuracy"); plt.xlabel("Epoch"); plt.legend(); plt.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('/kaggle/working/scratch_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()


**If `val_acc` is still climbing at epoch 30**, training hasn't converged — bump `N_EPOCHS` and rerun (it'll pick up faster since you now know per-epoch timing). If it plateaus early, that's fine — with only ~2 val utterances/speaker this curve is noisy by nature; the real signal is the verification/identification eval below, not this classification accuracy.

## 5. Evaluate the from-scratch model — same protocol as the baseline notebook

Reusing the exact same `veri_test2.txt` trials and `iden_split.txt` protocol as before, just swapping in this trained-from-scratch model for embedding extraction.

In [ ]:
embedding_model.eval()

embedding_cache_scratch = {}

@torch.no_grad()
def get_embedding_scratch(rel_path: str) -> torch.Tensor:
    if rel_path in embedding_cache_scratch:
        return embedding_cache_scratch[rel_path]
    full_path = resolve_wav(rel_path)
    signal, fs = torchaudio.load(str(full_path))
    if fs != SAMPLE_RATE:
        signal = torchaudio.functional.resample(signal, fs, SAMPLE_RATE)
    signal = signal.mean(dim=0, keepdim=True).to(device)  # [1, T]
    feats = extract_features(signal)
    emb = embedding_model(feats).squeeze().cpu()  # [EMB_DIM]
    embedding_cache_scratch[rel_path] = emb
    return emb

# Warm-up check
test_line = open(VERI_TEST_PATH).readline().split()
_ = get_embedding_scratch(test_line[1])
print("Embedding shape:", _.shape)


In [ ]:
# ---- Part 1: Verification ----
trials = []
with open(VERI_TEST_PATH) as f:
    for line in f:
        parts = line.strip().split()
        if len(parts) != 3:
            continue
        label, p1, p2 = parts
        trials.append((int(label), p1, p2))

N_TRIALS = 3000  # match whatever you used for the baseline for a fair comparison
random.seed(42)
eval_trials = trials if N_TRIALS is None else random.sample(trials, N_TRIALS)

unique_files = set()
for _, p1, p2 in eval_trials:
    unique_files.update([p1, p2])
for f in tqdm(unique_files, desc="Embedding test files"):
    get_embedding_scratch(f)

scores, labels = [], []
for label, p1, p2 in eval_trials:
    e1, e2 = get_embedding_scratch(p1), get_embedding_scratch(p2)
    sim = F.cosine_similarity(e1.unsqueeze(0), e2.unsqueeze(0)).item()
    scores.append(sim)
    labels.append(label)
scores, labels = np.array(scores), np.array(labels)

from sklearn.metrics import roc_curve

def compute_eer(labels, scores):
    fpr, tpr, thresholds = roc_curve(labels, scores, pos_label=1)
    fnr = 1 - tpr
    idx = np.nanargmin(np.abs(fpr - fnr))
    return (fpr[idx] + fnr[idx]) / 2, thresholds[idx]

def compute_min_dcf(labels, scores, p_target=0.05, c_miss=1, c_fa=1):
    fpr, tpr, thresholds = roc_curve(labels, scores, pos_label=1)
    fnr = 1 - tpr
    dcf = c_miss * fnr * p_target + c_fa * fpr * (1 - p_target)
    i = np.argmin(dcf)
    norm = min(c_miss * p_target, c_fa * (1 - p_target))
    return dcf[i] / norm

scratch_eer, _ = compute_eer(labels, scores)
scratch_min_dcf = compute_min_dcf(labels, scores)
print(f"Scratch SV  EER:    {scratch_eer*100:.2f}%")
print(f"Scratch SV  minDCF: {scratch_min_dcf:.4f}")


In [ ]:
# ---- Part 2: Identification ----
iden_entries = []
with open(IDEN_SPLIT_PATH) as f:
    for line in f:
        parts = line.strip().split()
        if len(parts) != 2:
            continue
        split_label, rel_path = parts
        speaker_id = rel_path.split('/')[0]
        iden_entries.append((int(split_label), rel_path, speaker_id))

train_entries = [e for e in iden_entries if e[0] == 1]
test_entries  = [e for e in iden_entries if e[0] == 3]

N_ENROLL_PER_SPEAKER = 3
N_TEST_PER_SPEAKER = 3
random.seed(42)

by_speaker_train = defaultdict(list)
for split, rel_path, spk in train_entries:
    by_speaker_train[spk].append(rel_path)
by_speaker_test = defaultdict(list)
for split, rel_path, spk in test_entries:
    by_speaker_test[spk].append(rel_path)

enroll_set = {spk: random.sample(files, min(N_ENROLL_PER_SPEAKER, len(files)))
              for spk, files in by_speaker_train.items()}
eval_set = {spk: random.sample(by_speaker_test[spk], min(N_TEST_PER_SPEAKER, len(by_speaker_test[spk])))
            for spk in by_speaker_test if spk in enroll_set}

speaker_centroids = {}
for spk, files in tqdm(enroll_set.items(), desc="Building centroids"):
    embs = torch.stack([get_embedding_scratch(f) for f in files])
    speaker_centroids[spk] = F.normalize(embs.mean(dim=0), dim=0)

centroid_ids = list(speaker_centroids.keys())
centroid_matrix = torch.stack([speaker_centroids[s] for s in centroid_ids])

correct, top5_correct, total = 0, 0, 0
for spk, files in tqdm(eval_set.items(), desc="Classifying"):
    for f in files:
        emb = F.normalize(get_embedding_scratch(f), dim=0)
        sims = centroid_matrix @ emb
        pred_spk = centroid_ids[torch.argmax(sims).item()]
        top5_spk = [centroid_ids[i] for i in torch.topk(sims, k=min(5, len(centroid_ids))).indices.tolist()]
        total += 1
        correct += int(pred_spk == spk)
        top5_correct += int(spk in top5_spk)

scratch_top1 = correct / total
scratch_top5 = top5_correct / total
print(f"Scratch SID Top-1: {scratch_top1*100:.2f}%  ({correct}/{total})")
print(f"Scratch SID Top-5: {scratch_top5*100:.2f}%  ({top5_correct}/{total})")


## 6. Compare against the baseline

If you saved `baseline_results.pkl` from the Week 1 notebook as a Kaggle Dataset output and attached it here, this loads it automatically. Otherwise, paste in the baseline numbers you printed earlier.

In [ ]:
baseline_candidates = list(Path("/kaggle/input").rglob("baseline_results.pkl"))
baseline = None
if baseline_candidates:
    with open(baseline_candidates[0], "rb") as f:
        baseline = pickle.load(f)
    print("Loaded baseline results from:", baseline_candidates[0])
else:
    print("baseline_results.pkl not found as an input — fill in the numbers manually below.")
    baseline = {
        'sv': {'eer': None, 'min_dcf': None},   # <-- paste your Week 1 numbers here if not auto-loaded
        'sid': {'top1_acc': None, 'top5_acc': None},
    }

print("\n=== SV/SID COMPARISON: Pretrained Baseline vs. Trained-from-Scratch (100 speakers) ===")
print(f"{'Metric':<15}{'Baseline':<15}{'From Scratch':<15}")
b_eer = baseline['sv']['eer']
b_mindcf = baseline['sv']['min_dcf']
b_top1 = baseline['sid']['top1_acc']
b_top5 = baseline['sid']['top5_acc']
print(f"{'EER':<15}{(f'{b_eer*100:.2f}%' if b_eer is not None else 'N/A'):<15}{f'{scratch_eer*100:.2f}%':<15}")
print(f"{'minDCF':<15}{(f'{b_mindcf:.4f}' if b_mindcf is not None else 'N/A'):<15}{f'{scratch_min_dcf:.4f}':<15}")
print(f"{'SID Top-1':<15}{(f'{b_top1*100:.2f}%' if b_top1 is not None else 'N/A'):<15}{f'{scratch_top1*100:.2f}%':<15}")
print(f"{'SID Top-5':<15}{(f'{b_top5*100:.2f}%' if b_top5 is not None else 'N/A'):<15}{f'{scratch_top5*100:.2f}%':<15}")

scratch_results = {
    'n_speakers_trained': N_SPEAKERS,
    'n_epochs': N_EPOCHS,
    'sv': {'eer': float(scratch_eer), 'min_dcf': float(scratch_min_dcf), 'n_trials': len(eval_trials)},
    'sid': {'top1_acc': float(scratch_top1), 'top5_acc': float(scratch_top5), 'n_test': total},
    'history': history,
}
with open('/kaggle/working/scratch_results.pkl', 'wb') as f:
    pickle.dump(scratch_results, f)
print("\nSaved scratch_results.pkl")


## Report-writing notes

- Expect the from-scratch model's EER to be **substantially higher** than the pretrained baseline (likely double digits vs. the baseline's ~1-2%) — that's the expected, correct outcome, not a bug. It's the direct consequence of ~10K utterances / 100 speakers vs. VoxCeleb2's ~1M utterances / 5,994 speakers used to train the pretrained model.
- This gap is itself the point of the comparison for your report: it demonstrates quantitatively why large-scale pretraining/transfer learning matters for this task, and explains why the app will use the pretrained (or fine-tuned) model rather than the from-scratch one — a legitimate, defensible design decision you should state explicitly.
- For the report's "training procedure" section, document: architecture (channels=512 config), loss (AAM-softmax, margin=0.2, scale=30), optimizer/schedule (Adam + OneCycleLR), epochs, batch size, crop length, and the compute budget/time this took — all of that is sitting in this notebook.
- Save this notebook's version as a Kaggle Dataset output (`scratch_results.pkl` + the final checkpoint) so you don't need to retrain when writing the report or wiring the model into the app later.
